## LIME explanations (5–10 test examples)
This notebook generates LIME explanations for a fixed set of test instances, using the trained MLP model from `artifacts/mlp/model.joblib`.
**Outputs** are saved to `artifacts/lime/` (HTML + JSON with weights).

Reproducibility notes:
- We fix `SEED` for both sampling test indices and for LIME itself.
- We persist chosen test indices to disk.

In [2]:
import sys

# Install LIME into the *current* Jupyter kernel environment.
# If installation succeeds but import still fails, restart the kernel and re-run.
!"{sys.executable}" -m pip install -q --upgrade pip
!"{sys.executable}" -m pip install -q lime
print("Python:", sys.executable)

Python: c:\Users\bestiary\AppData\Local\Programs\Python\Python312\python.exe


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.16.2 requires keras>=3.0.0, which is not installed.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 6.33.1 which is incompatible.
tensorflow-intel 2.16.2 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-intel 2.16.2 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.33.1 which is incompatible.
tensorflow-intel 2.16.2 requires tensorboard<2.17,>=2.16, but you have tensorboard 2.20.0 which is incompatible.
ultralytics 8.3.229 requires pillow<=12.0.0,>=7.1.2, but you have pillow 12.1.1 which is incompatible.


In [3]:
from pathlib import Path
import json

import joblib
import numpy as np
from lime.lime_tabular import LimeTabularExplainer


In [4]:
SEED = 42
N_EXAMPLES = 10  # set to 5..10 as needed
NUM_FEATURES = 10
NUM_SAMPLES = 5000

DATA_DIR = Path("data_processed")
MLP_DIR = Path("artifacts/mlp")
OUT_DIR = Path("artifacts/lime")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

meta = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
feature_names = meta["feature_names"]
class_names = meta["target_names"]

model = joblib.load(MLP_DIR / "model.joblib")

X_train.shape, X_test.shape, class_names[:]


((364, 30), (114, 30), ['malignant', 'benign'])

In [5]:
# Deterministically pick examples from the test set.
# We try to include both classes (if possible) for nicer reporting.

rng = np.random.default_rng(SEED)
idx0 = np.where(y_test == 0)[0]
idx1 = np.where(y_test == 1)[0]

half = N_EXAMPLES // 2
n0 = min(len(idx0), half)
n1 = min(len(idx1), N_EXAMPLES - n0)

chosen = []
if n0 > 0:
    chosen.extend(rng.choice(idx0, size=n0, replace=False).tolist())
if n1 > 0:
    chosen.extend(rng.choice(idx1, size=n1, replace=False).tolist())

# If class imbalance prevented us from reaching N_EXAMPLES, fill from remaining indices.
if len(chosen) < N_EXAMPLES:
    remaining = np.setdiff1d(np.arange(len(y_test)), np.array(chosen, dtype=int))
    extra = rng.choice(remaining, size=(N_EXAMPLES - len(chosen)), replace=False)
    chosen.extend(extra.tolist())

chosen = sorted(set(int(i) for i in chosen))
chosen[:5], len(chosen)


([7, 15, 49, 53, 55], 10)

In [6]:
# Save chosen indices (so SHAP/LEAF can reuse exactly the same instances)
indices_path = OUT_DIR / "chosen_test_indices.json"
indices_payload = {
    "seed": SEED,
    "n_examples": N_EXAMPLES,
    "chosen_test_indices": chosen,
}
indices_path.write_text(json.dumps(indices_payload, ensure_ascii=False, indent=2), encoding="utf-8")
indices_path.as_posix()


'artifacts/lime/chosen_test_indices.json'

In [7]:
explainer = LimeTabularExplainer(
    training_data=X_train,
    training_labels=y_train,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    discretize_continuous=True,
    random_state=SEED,
)

def predict_proba_fn(X):
    return model.predict_proba(X)


In [8]:
results = []
for i in chosen:
    x = X_test[i]
    true_y = int(y_test[i])
    proba = predict_proba_fn(x.reshape(1, -1))[0]
    pred_y = int(np.argmax(proba))

    exp = explainer.explain_instance(
        data_row=x,
        predict_fn=predict_proba_fn,
        num_features=NUM_FEATURES,
        num_samples=NUM_SAMPLES,
        top_labels=1,
    )

    # LIME returns feature ranges/conditions in strings; keep them as-is.
    # We store weights for the label that LIME explained.
    explained_label = int(exp.available_labels()[0])
    weights = exp.as_list(label=explained_label)

    html_path = OUT_DIR / f"lime_test_{i:03d}.html"
    html_path.write_text(exp.as_html(), encoding="utf-8")

    rec = {
        "test_index": int(i),
        "true_y": true_y,
        "pred_y": pred_y,
        "proba": [float(p) for p in proba],
        "explained_label": explained_label,
        "num_features": int(NUM_FEATURES),
        "num_samples": int(NUM_SAMPLES),
        "weights": [{"feature": f, "weight": float(w)} for f, w in weights],
        "html_file": html_path.name,
    }
    results.append(rec)

summary_path = OUT_DIR / "lime_summary.json"
summary_path.write_text(
    json.dumps(
        {
            "seed": SEED,
            "n_examples": len(results),
            "num_features": NUM_FEATURES,
            "num_samples": NUM_SAMPLES,
            "results": results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

summary_path.as_posix(), len(results)


('artifacts/lime/lime_summary.json', 10)